# Transfer Learning with ResNet-50 — Face Emotion Classification

**Model:** ResNet-50 (pretrained on ImageNet → fine-tuned)  
**Dataset:** FER2013 (Facial Expression Recognition) — 7 emotion classes  
**Classes:** angry, disgust, fear, happy, sad, surprise, neutral  
**Source:** [FER2013 on Kaggle](https://www.kaggle.com/datasets/msambare/fer2013) / HuggingFace

**Architecture:**
- ResNet-50 backbone (frozen initially → gradual unfreezing)
- Custom classifier head for 7 emotion classes

> GPU accelerator ဖွင့်ပြီး run ပါ: `Settings → Accelerator → GPU`

In [ ]:
# --- 1. Install Dependencies ---
# !pip install -q kaggle datasets torchvision scikit-learn seaborn tqdm

In [ ]:
# --- 2. Imports ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms, models
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import random
import os
import time
from tqdm import tqdm
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("✅ All imports successful.")

In [ ]:
# --- 3. Device & Hyperparameters ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Hyperparameters
IMG_SIZE = 224          # ResNet-50 standard input size
BATCH_SIZE = 32
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3    # classifier head LR
FINE_TUNE_LR = 1e-5     # backbone fine-tune LR
NUM_WORKERS = 2
FREEZE_EPOCHS = 5       # epochs to train with frozen backbone
NUM_CLASSES = 7

# Class names for FER2013
CLASS_NAMES = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")

## 📦 Dataset Download & Setup

FER2013 dataset ကို download လုပ်ပါမယ်။ **နည်းလမ်း ၂ ခု** ပေးထားပါတယ်:

1. **Option A** — Kaggle API (`kaggle datasets download`)
2. **Option B** — HuggingFace `datasets` library

တစ်ခုခုကိုပဲ run ပါ (default: Option B - HuggingFace)

In [ ]:
# --- 4A. Download FER2013 via Kaggle (uncomment if using Kaggle) ---
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/   # Upload your kaggle.json first
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d msambare/fer2013 -p ./data --unzip
# DATA_DIR = './data'

# --- 4B. Download FER2013 via HuggingFace (recommended) ---
from datasets import load_dataset

print("⏳ Downloading FER2013 from HuggingFace...")
hf_dataset = load_dataset("Piro17/fer2013")
print(f"✅ Download complete!")
print(f"Train samples: {len(hf_dataset['train'])}")
print(f"Test samples:  {len(hf_dataset['test'])}")

In [ ]:
# --- 5. Custom Dataset Class for FER2013 ---
class FER2013Dataset(Dataset):
    """PyTorch Dataset wrapper for FER2013 from HuggingFace."""
    def __init__(self, hf_split, transform=None):
        self.data = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # FER2013 images are 48x48 grayscale
        image = item['image'].convert('RGB')  # Convert grayscale → RGB for ResNet
        label = item['label']

        if self.transform:
            image = self.transform(image)

        return image, label

print("✅ FER2013Dataset class defined.")

In [ ]:
# --- 6. Data Transforms ---
# ImageNet normalization stats (ResNet was pretrained on ImageNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

print("✅ Transforms defined.")

In [ ]:
# --- 7. Create Datasets & DataLoaders ---
# Full training set → split into train (90%) + validation (10%)
full_train = FER2013Dataset(hf_dataset['train'], transform=train_transform)
test_dataset = FER2013Dataset(hf_dataset['test'], transform=val_test_transform)

# Split train → train + val
train_size = int(0.9 * len(full_train))
val_size = len(full_train) - train_size
train_dataset, val_dataset_raw = random_split(full_train, [train_size, val_size],
                                               generator=torch.Generator().manual_seed(SEED))

# Val dataset needs val_test_transform (no augmentation)
# We wrap it to override transform
class SubsetWithTransform(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        item = self.subset.dataset.data[self.subset.indices[idx]]
        image = item['image'].convert('RGB')
        label = item['label']
        if self.transform:
            image = self.transform(image)
        return image, label

val_dataset = SubsetWithTransform(val_dataset_raw, val_test_transform)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_dataset)} | Validation: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"Batches → Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

In [ ]:
# --- 8. Visualize Sample Images ---
def unnormalize(img_tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Reverse normalization for display."""
    img = img_tensor.clone()
    for t, m, s in zip(img, mean, std):
        t.mul_(s).add_(m)
    return torch.clamp(img, 0, 1)

# Show sample training images
fig, axes = plt.subplots(2, 7, figsize=(18, 6))
axes = axes.flatten()
for i in range(14):
    idx = random.randint(0, len(train_dataset) - 1)
    img, label = train_dataset[idx]
    img_display = unnormalize(img).permute(1, 2, 0).numpy()
    axes[i].imshow(img_display)
    axes[i].set_title(CLASS_NAMES[label], fontsize=10, fontweight='bold')
    axes[i].axis('off')
plt.suptitle("Sample Training Images (FER2013 — Face Emotions)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- 9. Class Distribution ---
# Count labels in training set
train_labels = []
for _, label in train_dataset:
    train_labels.append(label)

unique, counts = np.unique(train_labels, return_counts=True)
plt.figure(figsize=(10, 5))
bars = plt.bar([CLASS_NAMES[i] for i in unique], counts, color='steelblue', edgecolor='white')
for bar, c in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
             str(c), ha='center', fontsize=9)
plt.xlabel("Emotion Class")
plt.ylabel("Number of Samples")
plt.title("FER2013 Training Set — Class Distribution")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Check class imbalance
print("\nClass weights (inverse frequency) for handling imbalance:")
class_weights = 1.0 / torch.tensor(counts, dtype=torch.float32)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
for name, w in zip(CLASS_NAMES, class_weights):
    print(f"  {name:<10}: {w:.4f}")

## 🏗️ ResNet-50 Transfer Learning Model

**Strategy:**
1. **Phase 1 (Frozen backbone):** ResNet-50 backbone freeze လုပ်ပြီး classifier head ကိုပဲ train (5 epochs)
2. **Phase 2 (Fine-tune):** Backbone unfreeze လုပ်ပြီး low LR နဲ့ whole model ကို fine-tune

```
ResNet-50 (pretrained ImageNet)
    ↓
AdaptiveAvgPool2d → Flatten
    ↓
FC(2048 → 512) → BatchNorm → ReLU → Dropout(0.3)
    ↓
FC(512 → 128) → BatchNorm → ReLU → Dropout(0.3)
    ↓
FC(128 → 7)  ← 7 emotion classes
```

In [ ]:
# --- 10. Build ResNet-50 Transfer Learning Model ---
def build_resnet50(num_classes, pretrained=True):
    """Load pretrained ResNet-50 and replace classifier head."""
    # Load pretrained ResNet-50
    weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
    model = models.resnet50(weights=weights)

    # Freeze all backbone layers
    for param in model.parameters():
        param.requires_grad = False

    # Replace the final FC layer with custom classifier
    in_features = model.fc.in_features  # 2048 for ResNet-50
    model.fc = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(512, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(128, num_classes)
    )

    return model

model = build_resnet50(NUM_CLASSES, pretrained=True).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Total Parameters:     {total_params:>12,}")
print(f"Trainable Parameters: {trainable_params:>12,}")
print(f"Frozen Parameters:    {frozen_params:>12,}")
print(f"\nTrainable ratio: {trainable_params/total_params*100:.2f}%")

In [ ]:
# --- 11. Loss Function & Optimizer ---
# Weighted loss for class imbalance
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# Only optimize classifier head initially (backbone is frozen)
optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print("✅ Loss, Optimizer, Scheduler configured.")
print(f"   Phase 1 LR: {LEARNING_RATE} (classifier head only)")
print(f"   Phase 2 LR: {FINE_TUNE_LR} (full model fine-tune)")

In [ ]:
# --- 12. Training & Validation Functions ---
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="Training", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct/total:.4f}")

    return running_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / len(loader), correct / total

print("✅ train_one_epoch() and evaluate() defined.")

## 🚀 Training Loop

### Phase 1: Backbone Frozen (Feature Extraction Only)
ResNet-50 backbone ကို freeze ထားပြီး classifier head ကိုပဲ train လုပ်မယ်။

### Phase 2: Full Fine-Tuning
Backbone unfreeze လုပ်ပြီး low learning rate နဲ့ whole model ကို fine-tune လုပ်မယ်။

In [ ]:
# --- 13. Full Training Loop (2-Phase) ---
train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_val_acc = 0.0
lr_history = []

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    # ============ Phase 2: Unfreeze backbone after FREEZE_EPOCHS ============
    if epoch == FREEZE_EPOCHS:
        print("\n" + "=" * 60)
        print("🔓 Phase 2: Unfreezing backbone for fine-tuning!")
        print("=" * 60)

        # Unfreeze all parameters
        for param in model.parameters():
            param.requires_grad = True

        # New optimizer with differential learning rates
        optimizer = optim.Adam([
            {'params': model.conv1.parameters(),  'lr': FINE_TUNE_LR * 0.1},
            {'params': model.bn1.parameters(),    'lr': FINE_TUNE_LR * 0.1},
            {'params': model.layer1.parameters(), 'lr': FINE_TUNE_LR * 0.5},
            {'params': model.layer2.parameters(), 'lr': FINE_TUNE_LR * 0.5},
            {'params': model.layer3.parameters(), 'lr': FINE_TUNE_LR},
            {'params': model.layer4.parameters(), 'lr': FINE_TUNE_LR},
            {'params': model.fc.parameters(),     'lr': FINE_TUNE_LR * 5},
        ], weight_decay=1e-4)

        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"   Trainable parameters: {trainable:,}\n")

    # ============ Train & Evaluate ============
    phase = "Phase 1 (Frozen)" if epoch < FREEZE_EPOCHS else "Phase 2 (Fine-tune)"
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    # LR scheduling
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[-1]['lr']
    lr_history.append(current_lr)

    print(f"[{phase}] Epoch [{epoch+1}/{NUM_EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | LR: {current_lr:.2e}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_resnet50_fer2013.pth")
        print(f"  → 💾 Best model saved! (Val Acc: {val_acc:.4f})")

elapsed = time.time() - start_time
print(f"\n{'='*60}")
print(f"✅ Training Complete! Time: {elapsed/60:.1f} min")
print(f"   Best Validation Accuracy: {best_val_acc:.4f}")
print(f"{'='*60}")

In [ ]:
# --- 14. Plot Training History ---
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# (A) Loss
axes[0].plot(range(1, NUM_EPOCHS+1), train_losses, 'b-o', label='Train Loss', markersize=4)
axes[0].plot(range(1, NUM_EPOCHS+1), val_losses, 'r-o', label='Val Loss', markersize=4)
axes[0].axvline(x=FREEZE_EPOCHS+0.5, color='gray', linestyle='--', alpha=0.7, label='Unfreeze')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss per Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# (B) Accuracy
axes[1].plot(range(1, NUM_EPOCHS+1), train_accs, 'b-o', label='Train Accuracy', markersize=4)
axes[1].plot(range(1, NUM_EPOCHS+1), val_accs, 'r-o', label='Val Accuracy', markersize=4)
axes[1].axvline(x=FREEZE_EPOCHS+0.5, color='gray', linestyle='--', alpha=0.7, label='Unfreeze')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy per Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# (C) Learning Rate
axes[2].plot(range(1, NUM_EPOCHS+1), lr_history, 'g-o', markersize=4)
axes[2].axvline(x=FREEZE_EPOCHS+0.5, color='gray', linestyle='--', alpha=0.7, label='Unfreeze')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].set_yscale('log')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle("ResNet-50 Transfer Learning — Training History", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("resnet50_training_history.png", dpi=150, bbox_inches='tight')
plt.show()

## 🧪 Testing & Evaluation

Best model ကို load ပြန်လုပ်ပြီး Test set ပေါ်မှာ evaluate လုပ်ပါမယ်။

In [ ]:
# --- 15. Load Best Model & Test ---
model.load_state_dict(torch.load("best_resnet50_fer2013.pth", map_location=device, weights_only=True))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

test_acc = accuracy_score(all_labels, all_preds)
print(f"\n🎯 Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"\n{'='*60}")
print("Classification Report:")
print("=" * 60)
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

In [ ]:
# --- 16. Confusion Matrix ---
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('True', fontsize=12)
plt.title(f'Confusion Matrix — ResNet-50 on FER2013 (Acc: {test_acc:.2%})',
          fontsize=13, fontweight='bold')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("resnet50_confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 17. Per-Class Accuracy ---
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
per_class_acc = cm_normalized.diagonal()

plt.figure(figsize=(10, 5))
colors = ['#e74c3c', '#9b59b6', '#3498db', '#2ecc71', '#1abc9c', '#f39c12', '#e67e22']
bars = plt.bar(CLASS_NAMES, per_class_acc, color=colors, edgecolor='white')
plt.xlabel('Emotion Class', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Per-Class Accuracy — ResNet-50 on FER2013', fontsize=13, fontweight='bold')
plt.xticks(rotation=45)
plt.ylim(0, 1)
for bar, acc in zip(bars, per_class_acc):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{acc:.2%}', ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig("resnet50_per_class_accuracy.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 18. Show Random Predictions from Test Set ---
import torch.nn.functional as F

def show_emotion_predictions(model, dataset, class_names, num_images=15):
    model.eval()
    fig, axes = plt.subplots(3, 5, figsize=(18, 10))
    axes = axes.ravel()
    indices = random.sample(range(len(dataset)), num_images)

    with torch.no_grad():
        for i, idx in enumerate(indices):
            img, label = dataset[idx]
            output = model(img.unsqueeze(0).to(device))
            probs = F.softmax(output, dim=1).cpu().squeeze()
            _, pred = torch.max(output, 1)

            # Unnormalize for display
            img_display = unnormalize(img).permute(1, 2, 0).numpy()

            true_label = class_names[label]
            pred_label = class_names[pred.item()]
            confidence = probs[pred.item()].item()
            is_correct = true_label == pred_label

            axes[i].imshow(img_display)
            color = '#2ecc71' if is_correct else '#e74c3c'
            axes[i].set_title(f"True: {true_label}\nPred: {pred_label} ({confidence:.1%})",
                              color=color, fontsize=9, fontweight='bold')
            axes[i].axis('off')

    plt.suptitle("Test Set Predictions (🟢 Correct / 🔴 Wrong)",
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig("resnet50_predictions.png", dpi=150, bbox_inches='tight')
    plt.show()

show_emotion_predictions(model, test_dataset, CLASS_NAMES, num_images=15)

In [ ]:
# --- 19. Single Image Inference (Reusable) ---
def predict_emotion(model, image_path, class_names=CLASS_NAMES):
    """Predict emotion from a face image and display results."""
    model.eval()
    img_pil = Image.open(image_path).convert('RGB')
    img_tensor = val_test_transform(img_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(img_tensor)
        probs = F.softmax(logits, dim=1).cpu().squeeze().numpy()
        logits_np = logits.cpu().squeeze().numpy()
        pred_idx = np.argmax(probs)

    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Original image
    axes[0].imshow(img_pil)
    axes[0].set_title(f"Predicted: {class_names[pred_idx]} ({probs[pred_idx]:.1%})",
                      fontsize=12, fontweight='bold')
    axes[0].axis('off')

    # Logits
    colors = ['#3498db' if i == pred_idx else '#bdc3c7' for i in range(len(class_names))]
    axes[1].barh(class_names, logits_np, color=colors, edgecolor='white')
    axes[1].set_title('Raw Logits', fontsize=12, fontweight='bold')
    axes[1].axvline(x=0, color='gray', linewidth=0.8, linestyle='--')

    # Probabilities
    sorted_idx = np.argsort(probs)[::-1]
    axes[2].barh([class_names[i] for i in sorted_idx[::-1]],
                 probs[sorted_idx[::-1]],
                 color=['#2ecc71' if i == pred_idx else '#bdc3c7' for i in sorted_idx[::-1]],
                 edgecolor='white')
    axes[2].set_title('Softmax Probabilities', fontsize=12, fontweight='bold')
    axes[2].set_xlim(0, 1)

    plt.suptitle("Face Emotion Prediction — ResNet-50", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    return class_names[pred_idx], probs[pred_idx], probs

# Usage:
# predict_emotion(model, "path/to/face_image.jpg")
print("✅ predict_emotion() function ready. Usage:")
print('   predict_emotion(model, "path/to/face_image.jpg")')

In [ ]:
# --- 20. Save Final Model & Results ---
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': CLASS_NAMES,
    'num_classes': NUM_CLASSES,
    'img_size': IMG_SIZE,
    'architecture': 'resnet50',
    'best_val_acc': best_val_acc,
    'test_acc': test_acc,
    'train_history': {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accs': train_accs,
        'val_accs': val_accs,
        'lr_history': lr_history,
    }
}, "resnet50_fer2013_full.pth")

# Save results summary
with open("resnet50_results.txt", "w") as f:
    f.write("=" * 50 + "\n")
    f.write("ResNet-50 Transfer Learning — FER2013 Results\n")
    f.write("=" * 50 + "\n")
    f.write(f"Best Validation Accuracy: {best_val_acc:.4f}\n")
    f.write(f"Test Accuracy:            {test_acc:.4f}\n")
    f.write(f"Training Time:            {elapsed/60:.1f} min\n")
    f.write(f"Total Epochs:             {NUM_EPOCHS}\n")
    f.write(f"Freeze Epochs:            {FREEZE_EPOCHS}\n\n")
    f.write("Classification Report:\n")
    f.write(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

print("✅ Model & results saved!")
print(f"   - best_resnet50_fer2013.pth  (best weights)")
print(f"   - resnet50_fer2013_full.pth  (weights + metadata)")
print(f"   - resnet50_results.txt       (text report)")